# Google - The Gemma 4 Developer Agent Competition
## Notebook 02: Agent Evaluation & Benchmark Launcher

This launcher notebook executes the offline evaluation loop by importing the `pipeline` package:
1. Loads evaluation tasks.
2. Runs the local SWE-bench sandbox simulator.
3. Tests generated patches against validation test suites.
4. Computes the official **Pass@1** metric.

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline Import Bootstrap (Compatible with both Local and Kaggle Datasets)
# ---------------------------------------------------------------------------
import os
import sys
from pathlib import Path

# Detect environment and resolve pipeline location
LOCAL_ROOT = Path("..").resolve()
KAGGLE_INPUT = Path("/kaggle/input")

candidate_paths = [
    LOCAL_ROOT,
    LOCAL_ROOT / "pipeline",
    Path("."),
    Path("pipeline")
]

# Check Kaggle dataset mounts
if KAGGLE_INPUT.exists():
    for dataset_dir in KAGGLE_INPUT.glob("*"):
        if (dataset_dir / "pipeline").exists():
            candidate_paths.insert(0, dataset_dir)
        elif (dataset_dir / "main.py").exists() and "pipeline" in dataset_dir.name.lower():
            candidate_paths.insert(0, dataset_dir.parent)

pipeline_loaded = False
for p in candidate_paths:
    if (p / "pipeline" / "main.py").exists() or (p / "pipeline").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print(f"Successfully loaded pipeline from: {p}")
        pipeline_loaded = True
        break

if not pipeline_loaded:
    print("[WARN] Pipeline folder not auto-detected; defaulting to current sys.path.")

from pipeline.main import PipelineRunner, status, train, evaluate, package
runner = PipelineRunner()
runner.status()

### 1. Run Evaluation via Pipeline Controller
Single command execution using `runner.evaluate()` or `evaluate()`.

In [ ]:
# Execute evaluation on validation tasks
eval_metrics = runner.evaluate(
    tasks_path="../data/val_tasks.jsonl",
    max_tasks=50
)

print("\nFinal Leaderboard Metric Result:")
print(f"  • Total Tasks:  {eval_metrics.get('total_tasks')}")
print(f"  • Passed Tasks: {eval_metrics.get('passed')}")
print(f"  • Pass@1 Score: {eval_metrics.get('pass_rate_pct')}%")

### 2. Log Results to Experiments Tracker
Update your experiment log (`experiments/exp_XXX/notes.md`) with the observed pass rate and failure modes.